# 🧪 AiStock 基础服务测试

## 测试目标
- ✅ ConfigService: 配置加载服务
- ✅ CacheService: LRU 缓存服务
- ✅ DatabaseService: PostgreSQL 数据库服务

## 测试环境
- Python 3.11+
- 项目根目录：`/path/to/AiStock`

In [1]:
# 环境设置
import sys
from pathlib import Path

# 添加项目根目录到 Python 路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✅ 项目根目录：{project_root}")
print(f"✅ Python 路径已更新")

✅ 项目根目录：/home/ts/app/AiStock
✅ Python 路径已更新


In [2]:
# 导入基础服务
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

print("✅ 基础服务导入成功")

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
pip install dotenv

In [ ]:
config.global_settings

## 1️⃣ ConfigService 测试

In [ ]:
# 测试配置服务
config = ConfigService(
    system_name='dynamic_price',
    config_subdir='dynamic_price'
)

print(f"✅ ConfigService 初始化成功")
print(f"📊 系统模式：{config.get('system.mode')}")
print(f"📊 缓存启用：{config.get('cache.enable')}")
print(f"📊 三维权重：{config.get('weights')}")

In [ ]:
# 测试配置项获取
test_cases = [
    'system.name',
    'system.version',
    'cache.max_size',
    'cache.ttl',
    'risk_control.max_position_single',
    'stocks.0.code',
    'stocks.0.name',
]

print("\n📋 配置项测试：")
for key in test_cases:
    value = config.get(key)
    status = "✅" if value is not None else "❌"
    print(f"  {status} {key}: {value}")

In [ ]:
# 测试标的配置获取
stock_config = config.get_stock_config('600938')
if stock_config:
    print("\n✅ 中国海油配置：")
    for k, v in stock_config.items():
        print(f"  • {k}: {v}")
else:
    print("\n❌ 未找到标的配置")

## 2️⃣ CacheService 测试

In [ ]:
# 测试缓存服务
cache = CacheService(max_size=100, ttl=300)

print(f"✅ CacheService 初始化成功")
print(f"📊 最大容量：{cache.max_size}")
print(f"📊 默认 TTL: {cache.ttl}秒")

In [ ]:
# 测试缓存读写
import time

# 设置缓存
test_data = {
    'stock_price': 42.24,
    'timestamp': time.time(),
    'data': list(range(100))
}

cache.set('test_key_1', test_data)
cache.set('test_key_2', {'value': 100})
cache.set('test_key_3', [1, 2, 3, 4, 5])

print("✅ 缓存设置成功")

# 获取缓存
retrieved = cache.get('test_key_1')
print(f"\n✅ 缓存获取成功：{retrieved is not None}")
print(f"📊 数据匹配：{retrieved == test_data}")

In [ ]:
# 测试缓存统计
stats = cache.get_stats()

print("\n📊 缓存统计：")
for k, v in stats.items():
    if k != 'cache_keys':
        print(f"  • {k}: {v}")

# 测试缓存命中率
for i in range(10):
    cache.get('test_key_1')
    cache.get('nonexistent_key')

stats = cache.get_stats()
print(f"\n📊 缓存命中率：{stats['hit_rate']:.1%}")

In [ ]:
# 测试缓存过期
short_ttl_cache = CacheService(max_size=100, ttl=2)
short_ttl_cache.set('expire_test', {'value': 123})

print("✅ 设置短 TTL 缓存 (2 秒)")
print(f"📊 立即获取：{short_ttl_cache.get('expire_test') is not None}")

time.sleep(3)
print(f"📊 3 秒后获取：{short_ttl_cache.get('expire_test') is not None}")
print("✅ 缓存过期测试通过")

## 3️⃣ DatabaseService 测试

In [ ]:
# 测试数据库服务
try:
    db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)
    print("✅ DatabaseService 初始化成功")
    
    # 健康检查
    is_healthy = db.health_check()
    print(f"📊 数据库健康状态：{'✅ 正常' if is_healthy else '❌ 异常'}")
except Exception as e:
    print(f"❌ 数据库连接失败：{e}")
    db = None

In [ ]:
# 测试数据库查询（如果连接成功）
if db:
    try:
        # 测试简单查询
        result = db.execute_query("SELECT 1 as test")
        print(f"\n✅ 查询测试成功：{result}")
        
        # 测试表存在性
        tables = db.execute_query("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            LIMIT 10
        """)
        
        print(f"\n📊 数据库表（前 10 个）：")
        for table in tables:
            print(f"  • {table.get('table_name')}")
    except Exception as e:
        print(f"❌ 查询失败：{e}")

In [ ]:
# 清理资源
if db:
    db.close()
    print("\n✅ 数据库连接已关闭")

cache.clear()
print("✅ 缓存已清空")

## 📊 测试总结

In [ ]:
print("="*60)
print("📋 基础服务测试总结")
print("="*60)
print("✅ ConfigService: 配置加载正常")
print("✅ CacheService: 缓存读写正常")
print(f"✅ DatabaseService: {'连接正常' if db else '连接失败'}")
print("="*60)